In [1]:
import glob
import pandas as pd

# Load data

In [2]:
def load_evidence(fn):
    ds_name = fn.split("/")[-3]
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

In [3]:
fns = glob.glob("../../data/*/evidence_human/evidence.json")


In [4]:
dfs = []
for fn in fns:
    dfs.append(load_evidence(fn))
df = pd.concat(dfs)

# Perform token alignment

In [5]:
tdeg = df.query("source_type == 'text'").copy()

In [6]:
tdeg.head(2)

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,ds
0,text,Monocyte 1 strongly expressed FCGR3A (CD16) an...,text,homo_sapiens,Monocyte 1,kidney,none,FCGR3A,gene,kidney_Wu2018
1,text,"Interestingly, ABCA1, which mediates sterol ef...",text,homo_sapiens,Monocyte 1,kidney,none,ABCA1,gene,kidney_Wu2018


In [7]:
from extract.extract import align, norm_text

In [11]:
tdeg["source_cell_type_label"] = tdeg.apply(
    lambda x: align(norm_text(x["source_rationale"]), norm_text(x["extracted_cell_type_label"]))[0], 
    axis=1)
tdeg["source_feature_name"]    = tdeg.apply(
    lambda x: align(norm_text(x["source_rationale"]), norm_text(x["extracted_feature_name"]))[0], 
    axis=1)

In [12]:
tdeg["found_feature_name"] = tdeg["source_feature_name"] == tdeg["extracted_feature_name"]
tdeg["found_cell_type_label"] = tdeg["source_cell_type_label"] == tdeg["extracted_cell_type_label"]

In [18]:
nobs = tdeg.shape[0]
found_ct = tdeg["found_cell_type_label"].sum()
found_fn = tdeg["found_feature_name"].sum()
frac_ct = found_ct / nobs * 100
frac_fn = found_fn / nobs * 100


In [20]:
print(f"Features   found: {found_fn}/{nobs} ({frac_fn:.1f}%)")
print(f"Cell types found: {found_ct}/{nobs} ({frac_ct:.1f}%)")

Features   found: 160/1037 (15.4%)
Cell types found: 39/1037 (3.8%)
